In [185]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [224]:
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))

In [225]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [226]:
def build_dataset(words):
    block_size = 3
    X, Y = [], []
    
    for w in words:
        context = [0] * block_size
        
        for char in w + '.':
            ix = stoi[char]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [stoi[char]]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X, Y

In [227]:
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xtest, Ytest = build_dataset(words[n2:])

In [354]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 20), generator=g)
W1 = torch.randn((block_size*20, 300), generator=g)
b1 = torch.randn(300, generator=g)
W2 = torch.randn((300, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [355]:
for p in parameters:
    p.requires_grad = True

In [356]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [357]:
lri = []
lossi = []
stepi = []

for i in range(200000):
    ix = torch.randint(0, Xtr.shape[0], (100,), generator=g)
    
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, block_size*20) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    # if (i + 1) % 100 == 0:
    # print(loss.item())
    for p in parameters:
        p.grad = None
    loss.backward()
    
    #update
    if i <= 50000:
        lr = 0.1
    elif i <= 150000:
        lr = 0.05
    else:
        lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad
    # lri.append(lre[i])
    # lossi.append(loss.item())
    # stepi.append(i)

In [358]:
loss.item()

2.0424914360046387

In [360]:
emb = C[Xtr]
h = torch.tanh(emb.view(-1, block_size*20) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)

loss.item()

2.016162633895874

In [361]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, block_size*20) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)

loss.item()

2.15031099319458

In [389]:
for _ in range(10):
    out = []
    context = [0] * block_size

    while True:
        emb = C[torch.tensor(context)]
        h = torch.tanh(emb.view(-1, block_size*20) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))
        

phedin.
peca.
eilanitoelita.
lucian.
amirancuu.
haydeliya.
law.
bodej.
vianne.
kaiel.


In [ ]:
asmaine